# H&E Computational Pathology — WSI Adaptation

Adaptation of the OmicsLogic Applied Computational Pathology workflow for **whole-slide H&E images**.

**Environment:** `comp_path_workshop` (conda)

**Workflow:**  
1. Load SVS → 2. Tile-based Cellpose segmentation → 3. Feature extraction (morphology + Haralick) → 4. RF classification (Tumor vs TIL) → 5. Spatial analysis (nests, proximity)

**Design:**
- Self-contained (no modifications to existing project scripts)
- Tile-based processing for WSI (no full-slide load)
- Uses Cellpose as a library (inline calls only)

**Setup:**
```bash
conda env create -f envs/comp_path_workshop.yaml -n comp_path_workshop
conda activate comp_path_workshop
# Register kernel for Jupyter: python -m ipykernel install --user --name comp_path_workshop
```

## 1. Setup and configuration

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project root (parent of notebooks/)
PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()

# Paths
SVS_PATH = PROJECT_ROOT / "raw" / "PH 001 C13_085117.svs"
OUTPUT_DIR = PROJECT_ROOT / "output" / "he_comp_path"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Tiling parameters (match CellDIVE: 16384; level 0 = full res)
TILE_SIZE = 16384
OVERLAP = 1024
LEVEL = 0  # 0=full res (matches CellDIVE); 1+=downsampled

# Cellpose diameter: ~50 at 0.26 µm/px ≈ 13 µm nuclei
CELLPOSE_DIAMETER = 50

# Spatial analysis
NEST_DILATION_RADIUS = 15  # Pixels (reduced from 40 for bladder)
INTERACTION_RADIUS_PX = 50  # Tumor within this many px of TIL = "immune-engaged"

## 2. Dependencies and SVS loading

In [2]:
try:
    import openslide  # type: ignore[import-untyped]
    HAS_OPENSLIDE = True
except ImportError:
    HAS_OPENSLIDE = False

try:
    from cellpose import models  # type: ignore[import-untyped]
    HAS_CELLPOSE = True
except ImportError:
    HAS_CELLPOSE = False

try:
    import mahotas as mt  # type: ignore[import-untyped]
    HAS_MAHOTAS = True
except ImportError:
    HAS_MAHOTAS = False

if not HAS_OPENSLIDE:
    raise ImportError("openslide-python required. conda install -c conda-forge openslide-python")
if not HAS_CELLPOSE:
    raise ImportError("cellpose required. pip install cellpose")
if not HAS_MAHOTAS:
    raise ImportError("mahotas required. pip install mahotas")

if not SVS_PATH.exists():
    raise FileNotFoundError(f"SVS not found: {SVS_PATH}")

slide = openslide.OpenSlide(str(SVS_PATH))
dims = slide.level_dimensions[LEVEL]
print(f"SVS: {SVS_PATH.name}")
print(f"Level {LEVEL} dimensions: {dims[0]} x {dims[1]}")

SVS: PH 001 C13_085117.svs
Level 0 dimensions: 113415 x 87668


## 3. Tile grid and tissue detection

In [3]:
def get_tile_grid(width, height, tile_size, overlap):
    """Generate tile (x0, y0, w, h) at level dimensions."""
    step = tile_size - overlap
    for y0 in range(0, height - overlap, step):
        for x0 in range(0, width - overlap, step):
            w = min(tile_size, width - x0)
            h = min(tile_size, height - y0)
            if w > overlap and h > overlap:
                yield (x0, y0, w, h)

def read_tile(slide, x0, y0, w, h, level):
    """Read RGB tile from SVS. x0,y0 are in level coordinates; location converted to level-0."""
    downsample = slide.level_downsamples[level]
    loc = (int(x0 * downsample), int(y0 * downsample))
    region = slide.read_region(loc, level, (w, h))
    arr = np.array(region)
    if arr.shape[2] == 4:
        arr = arr[:, :, :3]
    return arr

def has_tissue(rgb, threshold=0.85):
    """True if tile mean intensity < threshold (darker = more tissue)."""
    gray = 0.299 * rgb[:,:,0] + 0.587 * rgb[:,:,1] + 0.114 * rgb[:,:,2]
    return np.mean(gray) < threshold * 255

## 4. Process one tile (Cellpose + features)

In [4]:
from skimage.color import rgb2gray
from skimage.measure import regionprops_table, regionprops

def process_tile(img_rgb, tile_x0, tile_y0, model, diameter=15):
    """
    Run Cellpose and extract morphology + Haralick per cell.
    Returns DataFrame with tile offset for global coordinates.
    """
    gray = (rgb2gray(img_rgb) * 255).astype(np.uint8)
    masks, _, _ = model.eval(img_rgb, diameter=diameter, channels=[0, 0])
    
    if masks.max() == 0:
        return pd.DataFrame()
    
    props = regionprops_table(masks, intensity_image=gray,
                              properties=['label', 'area', 'mean_intensity', 'eccentricity', 'centroid'])
    df = pd.DataFrame(props)
    
    regions = regionprops(masks, intensity_image=gray)
    haralick_list = []
    for r in regions:
        try:
            tex = mt.features.haralick(r.intensity_image.astype(np.uint8), ignore_zeros=True).mean(axis=0)
        except Exception:
            tex = np.zeros(13)
        haralick_list.append(tex)
    
    har_names = [f'haralick_{i}' for i in range(13)]
    df[har_names] = haralick_list
    df['global_x'] = df['centroid-1'] + tile_x0
    df['global_y'] = df['centroid-0'] + tile_y0
    return df

## 5. Tile-based WSI processing loop

In [5]:
from tqdm.auto import tqdm

print("Loading Cellpose nuclei model...")
try:
    model = models.CellposeModel(gpu=True, model_type='nuclei')
    print("Using GPU")
except Exception:
    model = models.CellposeModel(gpu=False, model_type='nuclei')
    print("Using CPU (GPU unavailable)")

width, height = dims[0], dims[1]
tiles = list(get_tile_grid(width, height, TILE_SIZE, OVERLAP))
print(f"Processing {len(tiles)} tiles...")

all_cells = []
for i, (x0, y0, w, h) in enumerate(tqdm(tiles)):
    img = read_tile(slide, x0, y0, w, h, LEVEL)
    if not has_tissue(img):
        continue
    df = process_tile(img, x0, y0, model, diameter=CELLPOSE_DIAMETER)
    if len(df) == 0:
        continue
    df['tile_id'] = i
    all_cells.append(df)

slide.close()

if len(all_cells) == 0:
    raise ValueError("No cells detected. Check SVS path, tile size, or tissue threshold.")

global_df = pd.concat(all_cells, ignore_index=True)
global_df['cell_id'] = np.arange(1, len(global_df) + 1)
print(f"Total cells: {len(global_df):,}")

model_type argument is not used in v4.0.1+. Ignoring this argument...


Loading Cellpose nuclei model...
Using GPU
Processing 48 tiles...


  0%|          | 0/48 [00:00<?, ?it/s]

channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
more than 65535 masks in image, masks returned as np.uint32
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
more than 65535 masks in image, masks returned as np.uint32
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
channels deprecated in v4.0.1+. If data contain more than 3 channels, only the first 3 channels will be used
more than 65535 masks in image, masks returned as np.uint32


Total cells: 317,206


## 6. Deduplicate cells in overlap regions

In [6]:
step = TILE_SIZE - OVERLAP
# Each tile 'owns' the region [x0, x0+step) x [y0, y0+step). Keep only cells whose centroid falls in that region.
keep = []
for i, (x0, y0, w, h) in enumerate(tiles):
    sub = global_df[global_df['tile_id'] == i]
    if len(sub) == 0:
        continue
    in_owned = (
        (sub['global_x'] >= x0) & (sub['global_x'] < x0 + step) &
        (sub['global_y'] >= y0) & (sub['global_y'] < y0 + step)
    )
    keep.extend(sub.index[in_owned].tolist())

global_df = global_df.loc[sorted(set(keep))].reset_index(drop=True)
global_df['cell_id'] = np.arange(1, len(global_df) + 1)
print(f"After deduplication: {len(global_df):,} cells")

After deduplication: 279,516 cells


## 7. EDA and RF classification

In [7]:
from sklearn.ensemble import RandomForestClassifier

feature_cols = ['area', 'mean_intensity', 'eccentricity'] + [f'haralick_{i}' for i in range(13)]
X = global_df[feature_cols]

# Simulated labels (tune for bladder: small/dark = TIL, large/pale = tumor)
area_thresh = 60
intensity_thresh = 100
y_sim = np.where((global_df['area'] < area_thresh) & (global_df['mean_intensity'] < intensity_thresh), 1, 0)

clf = RandomForestClassifier(n_estimators=100, random_state=42)
clf.fit(X, y_sim)
global_df['Predicted_Class'] = clf.predict(X)

til_count = int(global_df['Predicted_Class'].sum())
tumor_count = len(global_df) - til_count
print(f"Tumor cells: {tumor_count:,}")
print(f"TILs: {til_count:,}")

Tumor cells: 279,505
TILs: 11


## 8. Spatial analysis (nests + proximity)

In [8]:
from skimage.morphology import disk, binary_dilation
from scipy.ndimage import binary_fill_holes
from scipy.spatial import cKDTree

tumor_df = global_df[global_df['Predicted_Class'] == 0]
til_df = global_df[global_df['Predicted_Class'] == 1]
intra_tils = stromal_tils = engaged = isolated = 0  # defaults

if len(tumor_df) == 0 or len(til_df) == 0:
    print("Need both tumor and TIL cells for spatial analysis.")
else:
    scale = 10  # 1 raster px = 10 image px (reduce memory)
    rw = int(width / scale) + 1
    rh = int(height / scale) + 1
    raster = np.zeros((rh, rw), dtype=bool)
    ty = (tumor_df['global_y'] / scale).astype(int).clip(0, rh - 1)
    tx = (tumor_df['global_x'] / scale).astype(int).clip(0, rw - 1)
    raster[ty, tx] = True
    # Dilation on raster: radius ~ NEST_DILATION_RADIUS/scale for approx equivalence
    disk_r = max(1, NEST_DILATION_RADIUS // scale)
    nest_mask = binary_dilation(raster, disk(disk_r))
    nest_mask = binary_fill_holes(nest_mask)

    til_coords = til_df[['global_x', 'global_y']].values
    til_ry = (til_coords[:, 1] / scale).astype(int).clip(0, rh - 1)
    til_rx = (til_coords[:, 0] / scale).astype(int).clip(0, rw - 1)
    til_df = til_df.copy()
    til_df['Intratumoral'] = nest_mask[til_ry, til_rx]

    intra_tils = int(til_df['Intratumoral'].sum())
    stromal_tils = len(til_df) - intra_tils
    print(f"Intratumoral TILs: {intra_tils}")
    print(f"Stromal TILs: {stromal_tils}")

    til_tree = cKDTree(til_coords)
    tumor_coords = tumor_df[['global_x', 'global_y']].values
    dists, _ = til_tree.query(tumor_coords, k=1)
    tumor_df = tumor_df.copy()
    tumor_df['Interacting'] = dists <= INTERACTION_RADIUS_PX
    engaged = int(tumor_df['Interacting'].sum())
    isolated = len(tumor_df) - engaged
    print(f"Immune-engaged tumor cells: {engaged}")
    print(f"Isolated tumor cells: {isolated}")

Intratumoral TILs: 5
Stromal TILs: 6
Immune-engaged tumor cells: 69
Isolated tumor cells: 279436


## 9. Summary report

In [9]:
print("=" * 50)
print("H&E SPATIAL BIOLOGY REPORT (WSI)")
print("=" * 50)
print(f"Total cells: {len(global_df):,}")
print(f"Tumor cells: {tumor_count:,}")
print(f"TILs: {til_count:,}")
if len(tumor_df) > 0 and len(til_df) > 0:
    print(f"Intratumoral TILs: {intra_tils}")
    print(f"Stromal TILs: {stromal_tils}")
    print(f"Immune-engaged tumor cells: {engaged}")
    print(f"Isolated tumor cells: {isolated}")
print("=" * 50)

H&E SPATIAL BIOLOGY REPORT (WSI)
Total cells: 279,516
Tumor cells: 279,505
TILs: 11
Intratumoral TILs: 5
Stromal TILs: 6
Immune-engaged tumor cells: 69
Isolated tumor cells: 279436


## 10. Save outputs

In [10]:
global_df.to_csv(OUTPUT_DIR / "he_cells_classified.csv", index=False)
print(f"Saved: {OUTPUT_DIR / 'he_cells_classified.csv'}")

Saved: /home/steve/Projects/HeLab/BladderDIVE/output/he_comp_path/he_cells_classified.csv
